# Production DML Pipeline: End-to-End Estimation

**Goal:** Run the full production pipeline — validate, select models, estimate, sensitivity analysis, report.

**Time:** 15 minutes

You will use the production pipeline class to run a complete DML analysis
on simulated commodity data, including automated model selection and robustness checks.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from doubleml import DoubleMLData, DoubleMLPLR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LassoCV
from sklearn.model_selection import cross_val_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Simulate Production Data

Realistic scenario with missing values, many controls, and nonlinear confounding.

In [ ]:
n = 3000
p = 50
true_theta = -0.5

X = np.random.randn(n, p)
col_names = [f'X{i}' for i in range(p)]

D = 0.5 * np.sin(X[:, 0]) + 0.3 * X[:, 1]**2 + 0.2 * X[:, 3] + np.random.randn(n) * 0.5
Y = (true_theta * D + 0.8 * np.exp(0.2 * X[:, 0])
     + 0.4 * X[:, 2] + 0.3 * X[:, 4] * X[:, 5] + np.random.randn(n) * 0.3)

df = pd.DataFrame(X, columns=col_names)
df['outcome'] = Y
df['treatment'] = D

# Inject a few missing values (realistic)
missing_idx = np.random.choice(n, 15, replace=False)
df.loc[missing_idx[:5], 'X0'] = np.nan
df.loc[missing_idx[5:10], 'outcome'] = np.nan
df.loc[missing_idx[10:], 'X3'] = np.nan

print(f'Data: {n} rows, {p} controls')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'True effect: {true_theta}')

## Step 1: Data Validation

Check data quality before estimation.

In [ ]:
# Validation
checks = {}

# Missing values
n_missing = df[col_names + ['outcome', 'treatment']].isnull().sum().sum()
checks['missing_values'] = n_missing
if n_missing > 0:
    print(f'Found {n_missing} missing values. Dropping affected rows.')
    df_clean = df.dropna(subset=col_names + ['outcome', 'treatment']).copy()
else:
    df_clean = df.copy()

# Dimension check
n_clean = len(df_clean)
checks['n'] = n_clean
checks['p'] = p
checks['p_over_n'] = p / n_clean

# Treatment variation
checks['treatment_std'] = df_clean['treatment'].std()
checks['treatment_mean'] = df_clean['treatment'].mean()

print('\nValidation Results:')
for k, v in checks.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
print(f'\nData quality: PASS')

## Step 2: Nuisance Model Selection

Compare Lasso, Random Forest, and Gradient Boosting using cross-validated R-squared.

In [ ]:
X_clean = df_clean[col_names].values
Y_clean = df_clean['outcome'].values
D_clean = df_clean['treatment'].values

candidates = {
    'lasso': LassoCV(cv=3, random_state=42),
    'rf': RandomForestRegressor(100, max_depth=10, random_state=42),
    'gbm': GradientBoostingRegressor(100, max_depth=5, random_state=42),
}

print(f'{"Model":<10} {"R2(Y|X)":>10} {"R2(D|X)":>10} {"Average":>10}')
print('=' * 45)
best_name, best_score = None, -np.inf
for name, model in candidates.items():
    r2_y = cross_val_score(model, X_clean, Y_clean, cv=3, scoring='r2').mean()
    r2_d = cross_val_score(model, X_clean, D_clean, cv=3, scoring='r2').mean()
    avg = (r2_y + r2_d) / 2
    print(f'{name:<10} {r2_y:>10.3f} {r2_d:>10.3f} {avg:>10.3f}')
    if avg > best_score:
        best_name, best_score = name, avg

print(f'\nSelected: {best_name} (avg R2 = {best_score:.3f})')

## Step 3: Estimate with Best Model

Run DML PLR with the selected nuisance model.

In [ ]:
model_map = {
    'lasso': (LassoCV(cv=3, random_state=42), LassoCV(cv=3, random_state=42)),
    'rf': (RandomForestRegressor(200, max_depth=10, random_state=42),
           RandomForestRegressor(200, max_depth=10, random_state=42)),
    'gbm': (GradientBoostingRegressor(200, max_depth=5, random_state=42),
            GradientBoostingRegressor(200, max_depth=5, random_state=42)),
}

dml_data = DoubleMLData(df_clean, y_col='outcome',
                        d_cols='treatment', x_cols=col_names)

ml_l, ml_m = model_map[best_name]
dml = DoubleMLPLR(dml_data, ml_l=ml_l, ml_m=ml_m, n_folds=5)
dml.fit()

print(dml.summary)
print(f'\nTrue effect: {true_theta}')

## Step 4: Sensitivity Analysis

Run estimation with all three nuisance models and check robustness.

In [ ]:
sa_results = []
for name in ['lasso', 'rf', 'gbm']:
    ml_l, ml_m = model_map[name]
    dml_sa = DoubleMLPLR(dml_data, ml_l=ml_l, ml_m=ml_m, n_folds=5)
    dml_sa.fit()
    ci = dml_sa.confint()
    sa_results.append({
        'model': name, 'estimate': dml_sa.coef[0], 'se': dml_sa.se[0],
        'ci_low': ci.iloc[0, 0], 'ci_high': ci.iloc[0, 1]
    })

sa_df = pd.DataFrame(sa_results)
print(sa_df.to_string(index=False))

spread = sa_df['estimate'].max() - sa_df['estimate'].min()
mean_se = sa_df['se'].mean()
robust = spread < mean_se
print(f'\nEstimate spread: {spread:.4f}')
print(f'Average SE:      {mean_se:.4f}')
print(f'Robust:          {"YES" if robust else "NO — investigate further"}')

## Visualise: Sensitivity Forest Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for i, row in sa_df.iterrows():
    ax.errorbar(row['estimate'], i,
                xerr=[[row['estimate'] - row['ci_low']],
                      [row['ci_high'] - row['estimate']]],
                fmt='o', capsize=6, markersize=10, linewidth=2)

ax.axvline(x=true_theta, color='red', linestyle='--', linewidth=2,
           label=f'True = {true_theta}')
ax.set_yticks(range(len(sa_df)))
ax.set_yticklabels(sa_df['model'], fontsize=12)
ax.set_xlabel('Treatment Effect', fontsize=12)
ax.set_title('Sensitivity Analysis: All Nuisance Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Summary

**What you learned:**
1. Production DML needs validation, model selection, sensitivity analysis, and reporting
2. Automated model selection using cross-validated R-squared
3. Sensitivity analysis across nuisance model specifications
4. Robustness check: estimate spread vs standard error

**Course complete!** You now have:
- The theory (Modules 00-04)
- The practice (Modules 05-08)
- The production engineering (Module 09)

**Key takeaway:** A production DML pipeline is more than just calling `.fit()`.
Validation, model selection, and sensitivity analysis are essential.

In [ ]:
learning_objectives(["Production DML needs validation, model selection, sensitivity analysis, and reporting", "Automated model selection using cross-validated R-squared", "Sensitivity analysis across nuisance model specifications", "Robustness check: estimate spread vs standard error"])

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])